# 第 9 章の講義と実習　ブートストラップ

講義では，手元の 80 件だけから平均の変動を近似し，区間を作るまでを確かめます。後半の約 20 分は別の標本を一組得て，同じ作り方の区間がどう変わるか調べます。

## 準備

第 8 章では元の 342 件から何度も標本を選べました。今回は一組の 80 件だけを受け取ったら，その平均から母平均をどの幅で推定できるかが問いです。まず本文の標本で区間の作り方を確かめ，実習では同じモデルから得た別の 80 件で区間を作って，同じ軸で比べます。

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。

準備セルは実測記録 342 行を `model` に取ります。教材の仮想調査では，その各行を同じ確率で独立に復元抽出します。第 8 章の一組の標本内では同じ行を二度選ばない規則だったので，抽出の条件が違います。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

file_name = "penguins.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

penguins = pd.read_csv(csv_path)
has_measurements = (
    penguins["body_mass_g"].notna()
    & penguins["bill_length_mm"].notna()
    & penguins["bill_depth_mm"].notna()
    & penguins["flipper_length_mm"].notna()
)
model = penguins.loc[has_measurements]
len(model)

## 講義中に実行するセル

見出しの順に実行します。セルを動かす前に，比べる行・値・図を確認します。

### 1. 手元に届いた 80 件を確認する

本文と同じく，342 件のモデルからシード 0 で独立に復元抽出し，80 件の元標本を一組作ります。この後，区間を計算する担当者が使えるのはその 80 件だけです。表の一行は一回選ばれた体重，図の棒は体重の区間に入る件数だと確かめてください。

In [ ]:
lecture_sample = model.sample(n=80, replace=True, random_state=0, ignore_index=True)
lecture_mass = lecture_sample["body_mass_g"]
lecture_sample_mean = lecture_mass.mean()
print("Sample records:", len(lecture_sample), "Sample mean (g):", lecture_sample_mean)
lecture_sample[["body_mass_g"]].head(3)

In [ ]:
fig, ax = plt.subplots(layout="constrained")
ax.hist(lecture_mass, bins=range(2600, 6401, 200), color="#586c7a", edgecolor="white")
ax.axvline(lecture_sample_mean, color="#343a42", linestyle="--")
ax.set(xlabel="Body mass (g)", ylabel="Records", xlim=(2600, 6400))
plt.show()

### 2. 手元の 80 件から一回引き直す

本当に調査をやり直すならモデルから新しい 80 件を得ますが，担当者には元標本の `lecture_sample` しかありません。そこから同じ件数を復元で選び，経験分布を使った再標本を作ります。行ラベルを残した最初の 8 行で，元標本の同じ行が二度選ばれた箇所を探してください。

In [ ]:
lecture_resample = lecture_sample.sample(n=80, replace=True, random_state=0)
print("Records:", len(lecture_resample), "Different labels:", lecture_resample.index.nunique())
lecture_resample[["body_mass_g"]].head(8)

### 3. 再標本ごとに平均を一つ残す

同じ元標本 80 件から再標本を 2,000 組作り，各組の平均を一つずつ残します。先ほどの図は個体の体重 80 個を数えましたが，今度は 80 件ずつの平均 2,000 個を数えます。山が手元の標本平均を示す破線のどちら側に多いか，どれほど広がるかを読んでください。

In [ ]:
lecture_repetitions = 2000
lecture_boot_values = []
for lecture_seed in range(lecture_repetitions):
    lecture_resample = lecture_sample.sample(n=80, replace=True, random_state=lecture_seed)
    lecture_boot_values.append(lecture_resample["body_mass_g"].mean())
lecture_boot = pd.Series(lecture_boot_values)
print("Number of resample means:", len(lecture_boot), "First three (g):", lecture_boot[:3].to_list())

In [ ]:
fig, ax = plt.subplots(layout="constrained")
lecture_boot_bins = range(3700, 4651, 25)
ax.hist(lecture_boot, bins=lecture_boot_bins, color="#b15334", edgecolor="white")
ax.axvline(lecture_sample_mean, color="#343a42", linestyle="--")
ax.set(xlabel="Mean body mass (g)", ylabel="Resamples", xlim=(3700, 4650))
plt.show()

### 4. 調査を取り直したときの変動と比べる

手元の再標本平均が，本当に調査を取り直したときの平均の変動を近似するかを，教材の既知のモデルで点検します。上段には平均そのものを描きます。下段では，モデルから新しい標本を得た平均からモデルの母平均を引き，再標本平均からは手元の標本平均を引きます。中心の位置が違う二つの分布について，0 g の周りの広がりを比べてください。

In [ ]:
lecture_model_mean = model["body_mass_g"].mean()
lecture_model_values = []
for lecture_seed in range(lecture_repetitions):
    lecture_new_sample = model.sample(n=80, replace=True, random_state=lecture_seed)
    lecture_model_values.append(lecture_new_sample["body_mass_g"].mean())
lecture_model_means = pd.Series(lecture_model_values)

In [ ]:
fig, (average_ax, centered_ax) = plt.subplots(
    2, 1, sharey=True, figsize=(6.2, 5), layout="constrained",
)
average_ax.hist(lecture_model_means, bins=lecture_boot_bins, histtype="step", label="Model")
average_ax.hist(lecture_boot, bins=lecture_boot_bins, histtype="step",
                linestyle="--", label="Bootstrap")
average_ax.set(xlabel="Mean body mass (g)", ylabel="Runs", xlim=(3700, 4650))
average_ax.legend(frameon=False)
lecture_change_bins = range(-400, 401, 25)
centered_ax.hist(lecture_model_means - lecture_model_mean,
                 bins=lecture_change_bins, histtype="step")
centered_ax.hist(lecture_boot - lecture_sample_mean,
                 bins=lecture_change_bins, histtype="step", linestyle="--")
centered_ax.axvline(0, color="#343a42", linestyle=":")
centered_ax.set(xlabel="Difference from each center (g)", ylabel="Runs", xlim=(-400, 400))
plt.show()

### 5. 両端を求めて区間を読む

再標本平均を小さい順に並べ，`quantile()` で 2.5% 点と 97.5% 点を求めます。この両端を母平均の推定区間にするパーセンタイル法です。図の横線の両端はその値で，点は元標本の平均です。体重一件の範囲ではなく，平均の推定に使う幅として読んでください。

In [ ]:
lecture_interval = lecture_boot.quantile([0.025, 0.975])
lecture_lower, lecture_upper = lecture_interval
print(lecture_interval.round(2).to_string())

fig, ax = plt.subplots(figsize=(6.1, 1.9), layout="constrained")
ax.hlines(0, lecture_lower, lecture_upper, color="#286b70", linewidth=3)
ax.scatter(lecture_sample_mean, 0, color="#286b70", zorder=3)
ax.set(xlabel="Mean body mass (g)", yticks=[], xlim=(3700, 4500))
plt.show()

### 6. 一つの区間と，区間を作る方法

上の横線は，今回の元標本一組から作った区間です。この一本が母平均を含むかという結果と，区間を作る方法の被覆率は違います。被覆率を調べるには，モデルから元標本を何度も取り直し，各標本で区間を作って，固定された母平均を含む本数を数えます。本文の「区間を点検する」では，その 10,000 回の結果を読みます。

## 自分で分析する実習（約 20 分）

### 1. 別の 80 件を得たら（約 5 分）

本文の報告では，元標本を得るときのシードを `0` にしました。標本平均は約 4,054.06 g，名目 95% 区間は約 3,885.30〜4,225.97 g です。

別の担当者はシード `1` で元標本を得ます。元のモデル，独立復元抽出という方法，80 件という件数は本文と同じです。ただし，選ばれた体重の組が変わるため，標本平均と区間の端点は動き得ます。本文の値と比べて位置や幅がどうなりそうか，分からない点も含めて理由とともに一文で予想してください。

（ここに記入）

### 2. 新しい標本から区間を作る（約 9 分）

新しい元標本を得る一回だけ `model` を使います。シード `1` で 80 件を独立に復元抽出し，`sample` に入れる一行を書いてください。本文と同じ `ignore_index=True` で，担当者の手元にある 80 行へ `0` からラベルを付けます。

続くコードは，作った `sample` の中だけから 80 件ずつ復元で 2,000 回引き直し，各回の平均を `boot` に集めます。再標本化のシードは本文と同じ `0` から `1999` です。シードが同じでも元標本が違えば，再標本の平均は同じとは限りません。`sample` を記入するまでは出力は表示されません。

In [ ]:
sample = None
# ここにシード 1 の元標本を得る一行を書く。

boot = None
if sample is None:
    print("元標本を得る一行を書いてから，このセルを実行します。")
else:
    sample_mean = sample["body_mass_g"].mean()
    B = 2000
    means = []
    for seed in range(B):
        resample = sample.sample(n=len(sample), replace=True, random_state=seed)
        means.append(resample["body_mass_g"].mean())
    boot = pd.Series(means)
    print(len(sample), round(sample_mean, 2), len(boot))

左から元標本の件数，その 80 件の平均体重（g），再標本平均の個数が表示されます。`80` と `2000` を確かめてから，`boot` の 2.5% 点と 97.5% 点を `interval` に入れる一行を書いてください。分位点を取るのは個体の体重の列ではなく，再標本平均の列です。

横線の両端は区間の端点，点は元標本の平均です。横軸は二つの報告で同じです。区間を作る一行まで記入すると，比較図が表示されます。

In [ ]:
interval = None
if boot is not None:
    # ここに boot の二つの分位点を interval に入れる一行を書く。
    pass

if interval is None:
    print("元標本と区間の両方を計算すると，比較図が出ます。")
else:
    lower, upper = interval
    print(round(lower, 2), round(upper, 2), round(upper - lower, 2))

    fig, ax = plt.subplots(layout="constrained")
    ax.hlines(1, 3885.3046875, 4225.96875, color="#286b70", linewidth=2.3)
    ax.scatter(4054.0625, 1, color="#286b70", marker="o", s=40)
    ax.hlines(0, lower, upper, color="#b15334", linewidth=2.3)
    ax.scatter(sample_mean, 0, color="#b15334", marker="s", s=40)
    ax.set(xlabel="Mean body mass (g)", xlim=(3800, 4500), ylim=(-0.5, 1.5),
           yticks=[0, 1], yticklabels=["Seed 1", "Seed 0"])
    plt.show()

<details>
<summary>二つの行の書き方を確認する</summary>

`sample = None` の下で，元のモデルから標本を作ります。

```python
sample = model.sample(n=80, replace=True, random_state=1, ignore_index=True)
```

`boot` を作ったあと，次の行を `if boot is not None:` の中に書きます。

```python
    interval = boot.quantile([0.025, 0.975])
```

</details>

### 3. 二つの報告を比べる（約 6 分）

最初の予想と比較図を照らし，二つの担当者が得た結果を 2〜3 文で説明します。同じ母平均を推定するのに，元標本が違うと点と区間がどう動くかを答えてください。

1. 図のどこから，標本平均と区間の位置・幅の違いを読み取ったか。
2. 二つの区間は，同じ何の平均を推定しているか。標本サイズと方法を揃えても違う区間になったのはなぜか。

（ここに記入）

提出前にカーネルを再起動し，上から全セルを実行します。比較図と記入した説明を残したノートブックを保存します。

#### 90% の区間を試す（任意）

同じ `boot` から 5% 点と 95% 点を求め，元標本と再標本化を揃えたまま，名目 90% と 95% の幅を比べます。

#### 出典

Kristen Gorman と Palmer Station LTER の研究者による 2007〜2009 年の成鳥の計測を，Horst，Hill，Gorman が整理した表を使いました。本文とこの実習の独立復元抽出は教材で設定した調査のモデルです。[データの配布元](https://allisonhorst.github.io/palmerpenguins/) ・ [原著](https://doi.org/10.1371/journal.pone.0090081)